# 04 Fusion Pack Grouping — multi-category family/pack graph

Эта тетрадка идёт сразу после `03_matching_comparison.ipynb`. Здесь модели уже не запускаются: мы берём compact CSV с `score` и threshold-решениями, выбираем один fusion-run по `dev`, а затем строим два уровня графа для каждой категории.

Главное изменение: `04` больше не нужно запускать три раза руками. Если `03` сохранил общий multi-category отчёт, эта тетрадка сама разложит пары по `sauces`, `coconut_oil`, `soap` и сохранит отдельные `fusion_*` CSV для каждого run.

## Что здесь происходит

- `family` — один базовый товар. Например, `200 г` и `3 x 200 г` могут попасть в одну family, если модель решила, что это тот же продукт.
- `pack` — конкретная фасовка внутри family. Она отделяется простыми deterministic правилами по `unit_amount`, `total_amount`, `multipack_count`.
- По умолчанию финальный `family` теперь строится через Leiden community detection по weighted positive-рёбрам. Это снижает риск graph chaining: слабый bridge-edge не обязан склеивать всю connected component.
- Connected components сохраняются как audit-baseline (`connected_family_id`, `connected_pack_id`), а `fusion_family_id` / `fusion_pack_id` остаются финальным результатом этой тетрадки.
- Threshold выбирается только по `dev`. `test` используется как честная диагностика после выбора.
- Если в outputs из `03` нет `category_run`, notebook восстанавливает категорию через frozen split (`pair_key` -> `category_run`).


## 0. Локальные настройки

Обычно менять нужно только эту ячейку.

`MY_REPORTS_DIR_OVERRIDE` сейчас указывает на свежий общий output из `03`: `artifacts/reports/fine_tuning`. Если нужно вернуться к старым per-category отчётам, поставьте `None` — тогда каждый run будет читать свой стандартный reports folder.

`MY_CATEGORY_RUNS` можно оставить как есть: notebook сам пройдёт по трём текущим категориям и пропустит те, для которых в текущем reports-файле нет пар.

`MY_GRAPH_GROUPING_ALGORITHM = "leiden"` — рекомендуемый режим для family graph. Если нужен старый exact-baseline, поставьте `"connected_components"`.


In [ ]:
# === MY notebook settings ===
MY_CATEGORY_RUNS = ["sauces", "coconut_oil", "soap"]

# None = читать стандартные reports paths каждого category-run.
# Для нового frozen/fine-tuning benchmark из 03 используем общий reports folder.
MY_REPORTS_DIR_OVERRIDE = "artifacts/reports/fine_tuning"

# Нужен только если binary_threshold_predictions.csv не содержит category_run.
MY_EVAL_DATA_PATH = "research/dedup/data/training/dedup_pairs_final_pair_stratified_split.csv"

# None = выбрать method автоматически по dev. Можно поставить строку из binary_threshold_summary.csv.
MY_FUSION_METHOD = None

# None = сначала threshold_weighted_cost, если есть; иначе threshold_cost_sensitive.
MY_FUSION_THRESHOLD_STRATEGY = None

# "test" = честная диагностика после выбора на dev. Можно поставить "dev" или "all".
MY_FUSION_EVAL_SPLIT = "test"

# Если True, notebook упадёт, когда в общем reports-файле нет пар для одной из MY_CATEGORY_RUNS.
# Для smoke-output из 03 лучше оставить False.
MY_REQUIRE_ALL_CATEGORY_RUNS = False

# Graph grouping: "leiden" режет слабые bridge-связи внутри большой connected component.
# "connected_components" возвращает старое поведение один-в-один.
MY_GRAPH_GROUPING_ALGORITHM = "leiden"
MY_COMMUNITY_RESOLUTION = 1.0
MY_COMMUNITY_RANDOM_SEED = 42
MY_COMMUNITY_EDGE_WEIGHT_COL = "score"
MY_COMPARE_CONNECTED_COMPONENTS = True

# 3D graph view.
MY_3D_CATEGORY_RUN = "auto"  # "auto" = первая категория с сохранённым graph output
MY_3D_GRAPH = "family"  # "family" или "pack"
MY_3D_MAX_COMPONENTS = 8
MY_3D_MAX_NODES = 120


## Блок кода 1. Подготовка окружения

Импортируем только research helpers. Plotly опционален: если установлен, будет интерактивный 3D-граф; если нет, notebook построит статичный 3D-граф через matplotlib.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import math
import sys
from pathlib import Path
from typing import Any

import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    import plotly.graph_objects as go
except ImportError:
    go = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "research" / "dedup").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    ComponentConfig,
    GraphGroupingConfig,
    add_component_flags,
    build_graph_groups,
    component_size_summary,
    prepare_fusion_pair_edges,
    resolve_category_run,
    resolve_run_paths,
    select_fusion_run,
)

PROJECT_ROOT

## Блок кода 2. Пути и входные файлы

Если `MY_REPORTS_DIR_OVERRIDE` задан, все категории читают один общий `binary_threshold_summary.csv` / `binary_threshold_predictions.csv`. Это текущий основной режим после multi-category benchmark в `03`.

Если override пустой, каждая категория читает свой обычный reports folder.

In [ ]:
def _resolve_notebook_path(value: str | Path | None) -> Path | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    path = Path(text).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path.resolve()


CATEGORY_RUNS = [resolve_category_run(value) for value in MY_CATEGORY_RUNS]
REPORTS_DIR_OVERRIDE = _resolve_notebook_path(MY_REPORTS_DIR_OVERRIDE)
EVAL_DATA_PATH = _resolve_notebook_path(MY_EVAL_DATA_PATH)
EVAL_SPLIT = str(MY_FUSION_EVAL_SPLIT).strip().lower()
if EVAL_SPLIT not in {"test", "dev", "all"}:
    raise ValueError("MY_FUSION_EVAL_SPLIT должен быть 'test', 'dev' или 'all'")

GRAPH_GROUPING_CONFIG = GraphGroupingConfig(
    algorithm=str(MY_GRAPH_GROUPING_ALGORITHM).strip().lower(),
    edge_weight_col=str(MY_COMMUNITY_EDGE_WEIGHT_COL).strip() or "score",
    resolution=float(MY_COMMUNITY_RESOLUTION),
    seed=None if MY_COMMUNITY_RANDOM_SEED is None else int(MY_COMMUNITY_RANDOM_SEED),
)
CONNECTED_GROUPING_CONFIG = GraphGroupingConfig(algorithm="connected_components")
COMPARE_CONNECTED_COMPONENTS = bool(MY_COMPARE_CONNECTED_COMPONENTS)

input_rows = []
for run in CATEGORY_RUNS:
    paths = resolve_run_paths(PROJECT_ROOT, run)
    reports_dir = REPORTS_DIR_OVERRIDE or paths.reports_dir
    input_rows.append({
        "category_run": run.slug,
        "display_name": run.display_name,
        "project_name": run.project_name,
        "summary_path": reports_dir / "binary_threshold_summary.csv",
        "predictions_path": reports_dir / "binary_threshold_predictions.csv",
        "components_path": paths.fusion_components_path,
        "pair_eval_path": paths.fusion_pair_eval_path,
    })

input_plan = pd.DataFrame(input_rows)
display(input_plan)
print("Reports override:", REPORTS_DIR_OVERRIDE or "None -> per-category standard reports")
print("Eval data for category restore:", EVAL_DATA_PATH)
print("Diagnostic split:", EVAL_SPLIT)
print("Graph grouping:", GRAPH_GROUPING_CONFIG)
print("Connected-components audit:", COMPARE_CONNECTED_COMPONENTS)


## Блок кода 3. Загрузка reports и восстановление category_run

`03` может сохранить общий reports-файл без колонки `category_run`. Для split-файлов это нормально: ключ пары (`benchmark_pair_key`) совпадает с `pair_key` во frozen dataset, поэтому здесь мы аккуратно возвращаем категорию в predictions.

In [ ]:
@dataclass(frozen=True)
class LoadedReports:
    summary: pd.DataFrame
    predictions: pd.DataFrame
    summary_path: Path
    predictions_path: Path


_REPORT_CACHE: dict[tuple[Path, Path], LoadedReports] = {}


def _read_csv_cached(summary_path: Path, predictions_path: Path) -> LoadedReports:
    key = (summary_path, predictions_path)
    if key not in _REPORT_CACHE:
        if not summary_path.exists() or not predictions_path.exists():
            missing = [str(path) for path in [summary_path, predictions_path] if not path.exists()]
            raise FileNotFoundError("Missing reports from notebook 03: " + ", ".join(missing))
        _REPORT_CACHE[key] = LoadedReports(
            summary=pd.read_csv(summary_path),
            predictions=pd.read_csv(predictions_path),
            summary_path=summary_path,
            predictions_path=predictions_path,
        )
    return _REPORT_CACHE[key]


def _pair_key_column(frame: pd.DataFrame) -> str | None:
    for column in ["benchmark_pair_key", "pair_key"]:
        if column in frame.columns:
            return column
    return None


def _restore_category_run(predictions: pd.DataFrame, eval_data_path: Path | None) -> pd.DataFrame:
    output = predictions.copy()
    if "category_run" in output.columns and output["category_run"].notna().any():
        return output
    pair_key_col = _pair_key_column(output)
    if pair_key_col is None or eval_data_path is None or not eval_data_path.exists():
        output["category_run"] = pd.NA
        output["category_restore_status"] = "missing_pair_key_or_eval_data"
        return output

    eval_cols = pd.read_csv(eval_data_path, nrows=0).columns.tolist()
    wanted = [col for col in ["pair_key", "benchmark_pair_key", "category_run", "category_name", "project_name"] if col in eval_cols]
    eval_pairs = pd.read_csv(eval_data_path, usecols=wanted)
    eval_key_col = "pair_key" if "pair_key" in eval_pairs.columns else "benchmark_pair_key"
    if eval_key_col != pair_key_col:
        eval_pairs = eval_pairs.rename(columns={eval_key_col: pair_key_col})
    eval_pairs = eval_pairs.drop_duplicates(pair_key_col)

    merged = output.merge(eval_pairs, on=pair_key_col, how="left", suffixes=("", "_from_eval"))
    if "category_run_from_eval" in merged.columns:
        merged["category_run"] = merged["category_run"].fillna(merged["category_run_from_eval"])
        merged = merged.drop(columns=["category_run_from_eval"])
    merged["category_restore_status"] = merged["category_run"].notna().map({True: "ok", False: "unmatched_eval_pair"})
    return merged


load_status_rows: list[dict[str, Any]] = []
loaded_reports_by_key: dict[tuple[Path, Path], LoadedReports] = {}
for row in input_rows:
    try:
        loaded = _read_csv_cached(row["summary_path"], row["predictions_path"])
        restored = _restore_category_run(loaded.predictions, EVAL_DATA_PATH)
        loaded = LoadedReports(
            summary=loaded.summary,
            predictions=restored,
            summary_path=loaded.summary_path,
            predictions_path=loaded.predictions_path,
        )
        loaded_reports_by_key[(row["summary_path"], row["predictions_path"])] = loaded
        run_pairs = restored[restored["category_run"].astype(str).eq(row["category_run"])] if "category_run" in restored.columns else restored
        load_status_rows.append({
            "category_run": row["category_run"],
            "status": "loaded",
            "summary_rows": len(loaded.summary),
            "prediction_rows_total": len(restored),
            "prediction_rows_for_run": len(run_pairs),
            "category_restore": restored.get("category_restore_status", pd.Series(["already_present"])).value_counts().to_dict(),
        })
    except Exception as exc:
        load_status_rows.append({
            "category_run": row["category_run"],
            "status": "missing_or_failed",
            "error": str(exc),
        })

load_status = pd.DataFrame(load_status_rows)
display(load_status)

## Блок кода 4. Fusion helpers

Эти функции делают один и тот же шаг для каждой категории: выбрать run по `dev`, построить финальные Leiden/connected family groups, построить pack groups только внутри выбранной family, посчитать graph quality и сохранить compact CSV.


In [ ]:
FAMILY_EDGE_LABELS = {"exact_duplicate"}
PACK_EDGE_LABELS = {"exact_duplicate"}

pred_family_config = ComponentConfig(label_col="predicted_family_label", component_col="fusion_family_id")
pred_pack_config = ComponentConfig(label_col="predicted_pack_label", component_col="fusion_pack_id")
connected_family_config = ComponentConfig(label_col="predicted_family_label", component_col="connected_family_id")
connected_pack_config = ComponentConfig(label_col="predicted_pack_label", component_col="connected_pack_id")
true_family_config = ComponentConfig(label_col="true_family_label", component_col="true_family_id")
true_pack_config = ComponentConfig(label_col="true_pack_label", component_col="true_pack_id")


def _as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


def _node_catalog(pairs: pd.DataFrame, *, category_run: str) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for side in ["a", "b"]:
        mapping = {
            "node_id": f"raw_record_id_{side}",
            "marketplace": f"marketplace_{side}",
            "sku": f"sku_{side}",
            "brand": f"brand_{side}",
            "title": f"title_{side}",
            "unit_amount": f"unit_amount_{side}",
            "total_amount": f"total_amount_{side}",
            "multipack_count": f"multipack_count_{side}",
            "sales_volume": f"sales_volume_{side}",
        }
        present = {target: source for target, source in mapping.items() if source in pairs.columns}
        if "node_id" not in present:
            continue
        part = pairs[list(present.values())].rename(columns={source: target for target, source in present.items()})
        rows.extend(part.to_dict("records"))
    if not rows:
        return pd.DataFrame(columns=["category_run", "node_id"])
    output = pd.DataFrame(rows).dropna(subset=["node_id"]).drop_duplicates("node_id").reset_index(drop=True)
    output.insert(0, "category_run", category_run)
    return output


def _binary_link_report(frame: pd.DataFrame, *, true_col: str, pred_col: str, scope: str, category_run: str) -> dict[str, object]:
    true_link = _as_bool(frame[true_col]) if true_col in frame.columns else pd.Series(False, index=frame.index)
    pred_link = _as_bool(frame[pred_col]) if pred_col in frame.columns else pd.Series(False, index=frame.index)
    tp = int((true_link & pred_link).sum())
    fp = int((~true_link & pred_link).sum())
    fn = int((true_link & ~pred_link).sum())
    tn = int((~true_link & ~pred_link).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "category_run": category_run,
        "scope": scope,
        "eval_split": frame["split"].iloc[0] if "split" in frame.columns and frame["split"].nunique() == 1 else "all",
        "pairs": len(frame),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positive_links": tp,
        "false_links": fp,
        "missed_links": fn,
        "true_negative_links": tn,
    }


def _component_overview(components: pd.DataFrame, component_col: str, graph: str, category_run: str) -> dict[str, object]:
    sizes = component_size_summary(components, component_col=component_col)
    return {
        "category_run": category_run,
        "graph": graph,
        "components": int(sizes[component_col].nunique()) if not sizes.empty else 0,
        "multi_node_components": int((sizes["nodes"] > 1).sum()) if not sizes.empty else 0,
        "max_nodes": int(sizes["nodes"].max()) if not sizes.empty else 0,
    }


def _filter_summary_for_run(summary: pd.DataFrame, category_run: str) -> pd.DataFrame:
    if "category_run" not in summary.columns:
        return summary.copy()
    filtered = summary[summary["category_run"].astype(str).eq(category_run)].copy()
    return filtered if not filtered.empty else summary.copy()


def _filter_predictions_for_run(predictions: pd.DataFrame, category_run: str) -> pd.DataFrame:
    if "category_run" not in predictions.columns:
        return predictions.copy()
    return predictions[predictions["category_run"].astype(str).eq(category_run)].copy()


def _add_graph_metadata(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    output["fusion_grouping_algorithm"] = GRAPH_GROUPING_CONFIG.algorithm
    output["fusion_community_resolution"] = GRAPH_GROUPING_CONFIG.resolution
    output["fusion_community_seed"] = GRAPH_GROUPING_CONFIG.seed
    output["fusion_community_edge_weight_col"] = GRAPH_GROUPING_CONFIG.edge_weight_col
    return output


def _merge_components(base: pd.DataFrame, components: pd.DataFrame) -> pd.DataFrame:
    if components.empty:
        return base
    return base.merge(components, on="node_id", how="left")


def _build_and_save_one_run(row: dict[str, Any], loaded: LoadedReports) -> dict[str, Any]:
    category_run = str(row["category_run"])
    run_summary = _filter_summary_for_run(loaded.summary, category_run)
    run_predictions = _filter_predictions_for_run(loaded.predictions, category_run)
    if run_predictions.empty:
        if MY_REQUIRE_ALL_CATEGORY_RUNS:
            raise ValueError(f"No prediction rows for category_run={category_run!r}")
        return {"category_run": category_run, "status": "skipped_no_prediction_rows"}

    fusion_run = select_fusion_run(
        run_summary,
        method=MY_FUSION_METHOD,
        threshold_strategy=MY_FUSION_THRESHOLD_STRATEGY,
        split="dev",
    )
    fusion_pairs = prepare_fusion_pair_edges(run_predictions, fusion_run)

    pred_family_components = build_graph_groups(
        fusion_pairs,
        edge_labels=FAMILY_EDGE_LABELS,
        config=pred_family_config,
        grouping_config=GRAPH_GROUPING_CONFIG,
    )
    connected_family_components = build_graph_groups(
        fusion_pairs,
        edge_labels=FAMILY_EDGE_LABELS,
        config=connected_family_config,
        grouping_config=CONNECTED_GROUPING_CONFIG,
    ) if COMPARE_CONNECTED_COMPONENTS else pd.DataFrame(columns=["node_id", "connected_family_id"])
    true_family_components = build_graph_groups(
        fusion_pairs,
        edge_labels=FAMILY_EDGE_LABELS,
        config=true_family_config,
        grouping_config=CONNECTED_GROUPING_CONFIG,
    )

    family_flagged_pairs = add_component_flags(
        fusion_pairs,
        pred_family_components,
        config=pred_family_config,
        same_component_col="pred_same_family",
    )
    final_pack_mask = _as_bool(family_flagged_pairs["pred_same_family"])
    pred_pack_components = build_graph_groups(
        fusion_pairs,
        edge_labels=PACK_EDGE_LABELS,
        config=pred_pack_config,
        grouping_config=CONNECTED_GROUPING_CONFIG,
        edge_mask=final_pack_mask,
    )
    connected_pack_components = build_graph_groups(
        fusion_pairs,
        edge_labels=PACK_EDGE_LABELS,
        config=connected_pack_config,
        grouping_config=CONNECTED_GROUPING_CONFIG,
    ) if COMPARE_CONNECTED_COMPONENTS else pd.DataFrame(columns=["node_id", "connected_pack_id"])
    true_pack_components = build_graph_groups(
        fusion_pairs,
        edge_labels=PACK_EDGE_LABELS,
        config=true_pack_config,
        grouping_config=CONNECTED_GROUPING_CONFIG,
    )

    pair_eval = add_component_flags(fusion_pairs, true_family_components, config=true_family_config, same_component_col="true_same_family")
    pair_eval = add_component_flags(pair_eval, pred_family_components, config=pred_family_config, same_component_col="pred_same_family")
    pair_eval = add_component_flags(pair_eval, true_pack_components, config=true_pack_config, same_component_col="true_same_pack")
    pair_eval = add_component_flags(pair_eval, pred_pack_components, config=pred_pack_config, same_component_col="pred_same_pack")
    if COMPARE_CONNECTED_COMPONENTS:
        pair_eval = add_component_flags(
            pair_eval,
            connected_family_components,
            config=connected_family_config,
            same_component_col="connected_same_family",
        )
        pair_eval = add_component_flags(
            pair_eval,
            connected_pack_components,
            config=connected_pack_config,
            same_component_col="connected_same_pack",
        )
    if "category_run" in pair_eval.columns:
        pair_eval["category_run"] = category_run
    else:
        pair_eval.insert(0, "category_run", category_run)
    pair_eval = _add_graph_metadata(pair_eval)

    components_export = _node_catalog(fusion_pairs, category_run=category_run)
    for component_frame in [
        pred_family_components,
        pred_pack_components,
        true_family_components,
        true_pack_components,
        connected_family_components,
        connected_pack_components,
    ]:
        components_export = _merge_components(components_export, component_frame)
    components_export["fusion_method"] = fusion_run.method
    components_export["fusion_threshold_strategy"] = fusion_run.threshold_strategy
    components_export["fusion_threshold_same"] = fusion_run.threshold_same
    components_export = _add_graph_metadata(components_export)

    output_dir = Path(row["components_path"]).parent
    output_dir.mkdir(parents=True, exist_ok=True)
    components_export.to_csv(row["components_path"], index=False)
    pair_eval.to_csv(row["pair_eval_path"], index=False)

    if EVAL_SPLIT == "all" or "split" not in pair_eval.columns:
        eval_pairs = pair_eval.copy()
    else:
        eval_pairs = pair_eval[pair_eval["split"].astype(str).eq(EVAL_SPLIT)].copy()
        if eval_pairs.empty:
            eval_pairs = pair_eval.copy()

    selected_summary = run_summary[
        run_summary["method"].astype(str).eq(fusion_run.method)
        & run_summary["threshold_strategy"].astype(str).eq(fusion_run.threshold_strategy)
    ].copy()

    link_rows = [
        _binary_link_report(eval_pairs, true_col="true_same_family", pred_col="pred_same_family", scope="family", category_run=category_run),
        _binary_link_report(eval_pairs, true_col="true_same_pack", pred_col="pred_same_pack", scope="pack", category_run=category_run),
    ]
    component_rows = [
        _component_overview(pred_family_components, "fusion_family_id", "pred_family", category_run),
        _component_overview(pred_pack_components, "fusion_pack_id", "pred_pack", category_run),
        _component_overview(true_family_components, "true_family_id", "true_family_partial", category_run),
        _component_overview(true_pack_components, "true_pack_id", "true_pack_partial", category_run),
    ]
    if COMPARE_CONNECTED_COMPONENTS:
        link_rows.extend([
            _binary_link_report(
                eval_pairs,
                true_col="true_same_family",
                pred_col="connected_same_family",
                scope="connected_family_baseline",
                category_run=category_run,
            ),
            _binary_link_report(
                eval_pairs,
                true_col="true_same_pack",
                pred_col="connected_same_pack",
                scope="connected_pack_baseline",
                category_run=category_run,
            ),
        ])
        component_rows.extend([
            _component_overview(connected_family_components, "connected_family_id", "connected_family_baseline", category_run),
            _component_overview(connected_pack_components, "connected_pack_id", "connected_pack_baseline", category_run),
        ])

    link_report = pd.DataFrame(link_rows)
    component_report = pd.DataFrame(component_rows)

    return {
        "category_run": category_run,
        "status": "saved",
        "fusion_run": fusion_run,
        "selected_summary": selected_summary,
        "fusion_pairs": fusion_pairs,
        "pair_eval": pair_eval,
        "components": components_export,
        "link_report": link_report,
        "component_report": component_report,
        "components_path": Path(row["components_path"]),
        "pair_eval_path": Path(row["pair_eval_path"]),
        "pairs": len(pair_eval),
        "nodes": len(components_export),
        "family_edges": int(fusion_pairs["fusion_family_edge"].sum()),
        "pack_edges": int(fusion_pairs["fusion_pack_edge"].sum()),
        "grouping_algorithm": GRAPH_GROUPING_CONFIG.algorithm,
        "community_resolution": GRAPH_GROUPING_CONFIG.resolution,
        "community_seed": GRAPH_GROUPING_CONFIG.seed,
        "compare_connected_components": COMPARE_CONNECTED_COMPONENTS,
    }


## Блок кода 5. Запуск fusion для всех категорий

Эта ячейка — замена трём ручным запускам. Она сохраняет outputs в стандартные per-category пути, чтобы следующие notebooks и demo продолжали читать привычные `fusion_components_<suffix>.csv` / `fusion_pair_eval_<suffix>.csv`.

In [ ]:
fusion_outputs: dict[str, dict[str, Any]] = {}
status_rows: list[dict[str, Any]] = []
selection_rows: list[pd.DataFrame] = []
link_reports: list[pd.DataFrame] = []
component_reports: list[pd.DataFrame] = []

for row in input_rows:
    key = (row["summary_path"], row["predictions_path"])
    loaded = loaded_reports_by_key.get(key)
    if loaded is None:
        status_rows.append({"category_run": row["category_run"], "status": "skipped_missing_reports"})
        continue
    try:
        result = _build_and_save_one_run(row, loaded)
        fusion_outputs[str(row["category_run"])] = result
        status_rows.append({
            "category_run": row["category_run"],
            "status": result["status"],
            "method": getattr(result.get("fusion_run"), "method", None),
            "threshold_strategy": getattr(result.get("fusion_run"), "threshold_strategy", None),
            "threshold_same": getattr(result.get("fusion_run"), "threshold_same", None),
            "grouping_algorithm": result.get("grouping_algorithm"),
            "community_resolution": result.get("community_resolution"),
            "community_seed": result.get("community_seed"),
            "compare_connected_components": result.get("compare_connected_components"),
            "pairs": result.get("pairs", 0),
            "nodes": result.get("nodes", 0),
            "family_edges": result.get("family_edges", 0),
            "pack_edges": result.get("pack_edges", 0),
            "components_path": str(result.get("components_path", "")),
            "pair_eval_path": str(result.get("pair_eval_path", "")),
        })
        if result.get("status") == "saved":
            selected = result["selected_summary"].copy()
            selected.insert(0, "category_run", row["category_run"])
            selection_rows.append(selected)
            link_reports.append(result["link_report"])
            component_reports.append(result["component_report"])
    except Exception as exc:
        status_rows.append({"category_run": row["category_run"], "status": "failed", "error": str(exc)})

run_status = pd.DataFrame(status_rows)
selection_summary = pd.concat(selection_rows, ignore_index=True) if selection_rows else pd.DataFrame()
graph_quality = pd.concat(link_reports, ignore_index=True) if link_reports else pd.DataFrame()
component_summary_all = pd.concat(component_reports, ignore_index=True) if component_reports else pd.DataFrame()

display(run_status)

selection_cols = [
    "category_run",
    "method",
    "split",
    "threshold_strategy",
    "threshold_same",
    "precision",
    "recall",
    "f1",
    "weighted_f1",
    "false_merge_count",
    "false_split_count",
    "cost",
    "weighted_total_cost",
    "weight_source",
]
selection_cols = [column for column in selection_cols if column in selection_summary.columns]
if not selection_summary.empty:
    display(selection_summary[selection_cols].sort_values(["category_run", "split"]))

if not graph_quality.empty:
    display(graph_quality.round(4))

if not component_summary_all.empty:
    display(component_summary_all)

## Блок кода 6. Быстрые графики качества

Здесь не просто таблицы: смотрим, не появился ли слишком большой компонент и как family/pack качество отличается по категориям. Большой компонент — тревожный сигнал, потому что один false merge может сцепить много разных товаров. Если включён audit-baseline, таблицы также показывают старый connected-components результат рядом с финальным community detection.


In [ ]:
if plt is None:
    print("matplotlib is not available; skipping 2D charts")
elif run_status.empty or not (run_status["status"].astype(str) == "saved").any():
    print("No saved fusion outputs; skipping charts")
else:
    saved_status = run_status[run_status["status"].astype(str).eq("saved")].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].bar(saved_status["category_run"], saved_status["family_edges"], label="family")
    axes[0].bar(saved_status["category_run"], saved_status["pack_edges"], label="pack", alpha=0.75)
    axes[0].set_title("Positive graph edges")
    axes[0].set_ylabel("pairs")
    axes[0].legend()

    if not component_summary_all.empty:
        max_nodes = component_summary_all[component_summary_all["graph"].eq("pred_family")].copy()
        axes[1].bar(max_nodes["category_run"], max_nodes["max_nodes"])
    axes[1].set_title("Largest predicted family component")
    axes[1].set_ylabel("nodes")
    plt.tight_layout()
    plt.show()

if not graph_quality.empty and plt is not None:
    metric_frame = graph_quality[graph_quality["scope"].eq("family")].copy()
    if not metric_frame.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(metric_frame["category_run"], metric_frame["precision"], label="precision")
        ax.plot(metric_frame["category_run"], metric_frame["recall"], marker="o", label="recall")
        ax.set_ylim(0, 1.05)
        ax.set_title("Family graph quality on diagnostic split")
        ax.legend()
        plt.tight_layout()
        plt.show()

## Блок кода 7. Примеры ошибок и разных фасовок

Эта часть частично заменяет старый `05`: сразу после сборки графа видно, где модель склеила лишнее, где пропустила связь, и где family совпала, но pack отличается.

In [ ]:
example_cols = [
    "category_run",
    "split",
    "score",
    "same_base_product",
    "predicted_binary",
    "same_pack_signature",
    "title_a",
    "title_b",
    "brand_a",
    "brand_b",
    "unit_amount_a",
    "unit_amount_b",
    "total_amount_a",
    "total_amount_b",
    "multipack_count_a",
    "multipack_count_b",
    "pair_weight",
]

for category_run, result in fusion_outputs.items():
    if result.get("status") != "saved":
        continue
    pair_eval = result["pair_eval"]
    available_cols = [column for column in example_cols if column in pair_eval.columns]
    if EVAL_SPLIT == "all" or "split" not in pair_eval.columns:
        eval_pairs = pair_eval.copy()
    else:
        eval_pairs = pair_eval[pair_eval["split"].astype(str).eq(EVAL_SPLIT)].copy()
        if eval_pairs.empty:
            eval_pairs = pair_eval.copy()

    print(f"\n=== {category_run}: false family links ===")
    display(eval_pairs[~_as_bool(eval_pairs["true_same_family"]) & _as_bool(eval_pairs["pred_same_family"])][available_cols].head(10))

    print(f"\n=== {category_run}: missed family links ===")
    display(eval_pairs[_as_bool(eval_pairs["true_same_family"]) & ~_as_bool(eval_pairs["pred_same_family"])][available_cols].head(10))

    print(f"\n=== {category_run}: same family, different pack ===")
    display(pair_eval[_as_bool(pair_eval["pred_same_family"]) & ~_as_bool(pair_eval["pred_same_pack"])][available_cols].head(10))

## Блок кода 8. 3D graph visualization

Здесь строится 3D-карта крупнейших финальных family/pack groups. Точки — SKU, линии — positive edges внутри выбранной group, высота примерно отражает продажи или позицию внутри компонента. Это не метрика, а визуальная проверка: быстро видно, если одна family вдруг превращается в огромный комок.


In [ ]:
def _pick_3d_category(outputs: dict[str, dict[str, Any]], requested: str) -> str | None:
    saved = [key for key, value in outputs.items() if value.get("status") == "saved" and not value.get("components", pd.DataFrame()).empty]
    if not saved:
        return None
    if requested != "auto" and requested in saved:
        return requested
    return saved[0]


def _component_col_for_graph(graph: str) -> tuple[str, str]:
    graph = str(graph).strip().lower()
    if graph == "pack":
        return "fusion_pack_id", "pred_same_pack"
    return "fusion_family_id", "pred_same_family"


def _build_3d_layout(components: pd.DataFrame, pair_eval: pd.DataFrame, *, graph: str, max_components: int, max_nodes: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    component_col, same_col = _component_col_for_graph(graph)
    if components.empty or component_col not in components.columns:
        return pd.DataFrame(), pd.DataFrame()

    sizes = components.groupby(component_col, dropna=True).size().rename("nodes").reset_index()
    sizes = sizes[sizes["nodes"].gt(1)].sort_values(["nodes", component_col], ascending=[False, True]).head(max_components)
    if sizes.empty:
        return pd.DataFrame(), pd.DataFrame()

    nodes = components[components[component_col].isin(set(sizes[component_col]))].copy()
    nodes = nodes.merge(sizes, on=component_col, how="left")
    nodes = nodes.sort_values(["nodes", component_col, "node_id"], ascending=[False, True, True]).head(max_nodes).copy()
    node_ids = set(nodes["node_id"].astype(str))

    coords = []
    for component_index, (component_id, group) in enumerate(nodes.groupby(component_col, sort=True)):
        group = group.sort_values(["sales_volume", "node_id"], ascending=[False, True]) if "sales_volume" in group.columns else group.sort_values("node_id")
        n = max(len(group), 1)
        radius = 1.0 + math.log1p(n)
        center_x = (component_index % 4) * 5.0
        center_y = (component_index // 4) * 5.0
        for local_index, (_, row) in enumerate(group.iterrows()):
            angle = 2 * math.pi * local_index / n
            sales = pd.to_numeric(pd.Series([row.get("sales_volume")]), errors="coerce").fillna(0).iloc[0]
            total_amount = pd.to_numeric(pd.Series([row.get("total_amount")]), errors="coerce").fillna(0).iloc[0]
            z = math.log1p(float(sales)) if sales > 0 else float(local_index % 7)
            coords.append({
                "node_id": str(row["node_id"]),
                "component_id": component_id,
                "component_nodes": int(row["nodes"]),
                "x": center_x + radius * math.cos(angle),
                "y": center_y + radius * math.sin(angle),
                "z": z,
                "marker_size": max(5.0, min(22.0, 5.0 + math.log1p(float(sales)) * 2.0)),
                "label": f"{row.get('brand', '')} | {row.get('title', '')}"[:220],
                "sales_volume": sales,
                "total_amount": total_amount,
                "pack_id": row.get("fusion_pack_id"),
                "family_id": row.get("fusion_family_id"),
            })
    node_layout = pd.DataFrame(coords)
    if node_layout.empty:
        return node_layout, pd.DataFrame()

    coord_map = node_layout.set_index("node_id")[["x", "y", "z"]].to_dict("index")
    edge_rows = pair_eval.copy()
    if same_col in edge_rows.columns:
        edge_rows = edge_rows[_as_bool(edge_rows[same_col])]
    edge_rows = edge_rows[
        edge_rows["raw_record_id_a"].astype(str).isin(node_ids)
        & edge_rows["raw_record_id_b"].astype(str).isin(node_ids)
    ].copy()
    edges = []
    for row in edge_rows[["raw_record_id_a", "raw_record_id_b"]].drop_duplicates().itertuples(index=False):
        left = coord_map.get(str(row.raw_record_id_a))
        right = coord_map.get(str(row.raw_record_id_b))
        if left and right:
            edges.append({
                "x": [left["x"], right["x"], None],
                "y": [left["y"], right["y"], None],
                "z": [left["z"], right["z"], None],
            })
    return node_layout, pd.DataFrame(edges)


selected_3d_category = _pick_3d_category(fusion_outputs, str(MY_3D_CATEGORY_RUN))
if selected_3d_category is None:
    print("No saved fusion outputs for 3D graph")
else:
    selected_output = fusion_outputs[selected_3d_category]
    graph_nodes, graph_edges = _build_3d_layout(
        selected_output["components"],
        selected_output["pair_eval"],
        graph=MY_3D_GRAPH,
        max_components=int(MY_3D_MAX_COMPONENTS),
        max_nodes=int(MY_3D_MAX_NODES),
    )
    print(f"3D graph category={selected_3d_category}, graph={MY_3D_GRAPH}, nodes={len(graph_nodes)}, edges={len(graph_edges)}")

    if graph_nodes.empty:
        print("No multi-node components to visualize")
    elif go is not None:
        edge_x, edge_y, edge_z = [], [], []
        for row in graph_edges.itertuples(index=False):
            edge_x.extend(row.x)
            edge_y.extend(row.y)
            edge_z.extend(row.z)
        fig = go.Figure()
        if edge_x:
            fig.add_trace(go.Scatter3d(
                x=edge_x,
                y=edge_y,
                z=edge_z,
                mode="lines",
                line=dict(color="rgba(120,120,120,0.45)", width=3),
                hoverinfo="skip",
                name="positive edges",
            ))
        fig.add_trace(go.Scatter3d(
            x=graph_nodes["x"],
            y=graph_nodes["y"],
            z=graph_nodes["z"],
            mode="markers",
            text=graph_nodes["label"],
            hovertemplate=(
                "<b>%{text}</b><br>component=%{customdata[0]}<br>"
                "family=%{customdata[1]}<br>pack=%{customdata[2]}<br>"
                "sales=%{customdata[3]}<br>total_amount=%{customdata[4]}<extra></extra>"
            ),
            customdata=graph_nodes[["component_id", "family_id", "pack_id", "sales_volume", "total_amount"]],
            marker=dict(
                size=graph_nodes["marker_size"],
                color=graph_nodes["component_nodes"],
                colorscale="Turbo",
                opacity=0.9,
                colorbar=dict(title="component<br>size"),
            ),
            name="SKU nodes",
        ))
        fig.update_layout(
            title=f"3D {MY_3D_GRAPH} graph — {selected_3d_category}",
            height=780,
            scene=dict(
                xaxis_title="component layout x",
                yaxis_title="component layout y",
                zaxis_title="log sales / local height",
            ),
            margin=dict(l=0, r=0, t=50, b=0),
        )
        fig.show()
    elif plt is not None:
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection="3d")
        for row in graph_edges.itertuples(index=False):
            ax.plot(row.x[:2], row.y[:2], row.z[:2], color="0.65", linewidth=1, alpha=0.55)
        scatter = ax.scatter(
            graph_nodes["x"],
            graph_nodes["y"],
            graph_nodes["z"],
            s=graph_nodes["marker_size"] * 9,
            c=graph_nodes["component_nodes"],
            cmap="turbo",
            alpha=0.9,
        )
        ax.set_title(f"3D {MY_3D_GRAPH} graph — {selected_3d_category}")
        ax.set_xlabel("component layout x")
        ax.set_ylabel("component layout y")
        ax.set_zlabel("log sales / local height")
        fig.colorbar(scatter, ax=ax, shrink=0.65, label="component size")
        plt.tight_layout()
        plt.show()
    else:
        display(graph_nodes.head(50))

## Что делать после этой тетрадки

- Для быстрого research-вывода этого notebook уже достаточно: он строит groups, сохраняет CSV, показывает graph quality и 3D-компоненты.
